In [1]:
import polars as pl

salmonella_clusters = pl.read_csv('/gscratch/scrubbed/carsonjm/2026_03_05_uhbdb_2/92/3007084c40d21b6d0fedb5c0fa00bf/Salmonella_self_unique_cdhit.tsv', separator='\t')

In [2]:
salmonella_clusters.group_by('cluster').len().sort('len', descending=True)

cluster,len
str,u32
"""MOTUSDB_2026_03_04_RSGB23-1_GC…",1897
"""MOTUSDB_2026_03_04_RSGB23-1_GC…",1638
"""MOTUSDB_2026_03_04_RSGB23-1_GC…",1194
"""MOTUSDB_2026_03_04_RSGB23-1_GC…",1111
"""MOTUSDB_2026_03_04_RSGB23-1_GC…",1099
…,…
"""MOTUSDB_2026_03_04_RSGB23-1_GC…",1
"""MOTUSDB_2026_03_04_RSGB23-1_GC…",1
"""MOTUSDB_2026_03_04_RSGB23-1_GC…",1


In [3]:
cluster_id = salmonella_clusters.group_by('cluster').len().sort('len', descending=True)[0,0]

In [ ]:
(
    salmonella_clusters
        .filter(pl.col('cluster') == cluster_id)
        [['object']]
        .with_columns([
            pl.col('object').str.split('RSGB23-1_').list[-1].str.split('-V1').list[0].str.replace('-', '_')
        ])
        .write_csv('salmonella_test_samples.txt', separator='\t', include_header=False)
)

In [ ]:
%%bash
### Download salmonella genomes in the largest cluster
datasets download genome accession --inputfile salmonella_test_samples.txt --dehydrated

unzip ncbi_dataset.zip -d salmonella_tmp

datasets rehydrate \
    --directory salmonella_tmp \
    --gzip \
    --max-workers 4

In [ ]:
%%bash
### Run skani on salmonella genes in the largest cluster
skani triangle \
    /mmfs1/gscratch/pedslabs_hoffman/carsonjm/CFPhageome/repos/uhvdb-manuscript/uhbdb_creation/Salmonella_test/salmonella_tmp/ncbi_dataset/data/*/*_genomic.fna.gz \
    --ci \
    -E \
    --fast \
    -t 16 \
    -o salmonella_test_skani.tsv

In [1]:
import polars as pl

salmonella_test = (
    pl.read_csv('/mmfs1/gscratch/pedslabs_hoffman/carsonjm/CFPhageome/repos/uhvdb-manuscript/uhbdb_creation/Salmonella_test/salmonella_test_skani.tsv', separator='\t', columns=['ANI', 'Align_fraction_query', 'Align_fraction_ref', 'Ref_name', 'Query_name'])
)

In [4]:
salmonella_test.unique('Query_name')['ANI'].describe()

statistic,value
str,f64
"""count""",1896.0
"""null_count""",0.0
"""mean""",99.990005
"""std""",0.000398
"""min""",99.98
"""25%""",99.99
"""50%""",99.99
"""75%""",99.99
"""max""",100.0


In [5]:
salmonella_test.unique('Query_name')['Align_fraction_query'].describe()

statistic,value
str,f64
"""count""",1896.0
"""null_count""",0.0
"""mean""",99.477495
"""std""",1.006167
"""min""",95.15
"""25%""",99.79
"""50%""",99.84
"""75%""",99.88
"""max""",100.0


In [6]:
salmonella_test.unique('Query_name')['Align_fraction_ref'].describe()

statistic,value
str,f64
"""count""",1896.0
"""null_count""",0.0
"""mean""",99.865047
"""std""",0.064976
"""min""",99.23
"""25%""",99.83
"""50%""",99.86
"""75%""",99.89
"""max""",100.0
